# Preprocessing
what all we will do

- Combine ticket subject + ticket description into one column to have both subject for clean topic of ticket and description will add context to it. Model will require only one column then

- lowercase emntire text

- strip extra white spaces for better tockenization

- encode ticket type like "technical issue" into 0 and "biilong inquiry" to 1 as the model works with numbers, not strings

- Encode ticket priority to numbers the same way as ticket tye

- save the clean dataframe

In [85]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import re  ##regex to remobe the place holders

In [86]:
df = pd.read_csv('../Dataset/sample_tickets_dataset.csv')
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Ryan Allen,allen37@icloud.com,61,Female,CoreManage,29-07-2022,Refund Request,Service outage credit or refund request,The device stopped working completely after ex...,Resolved,Refund processed within 5-7 business days to o...,Low,Social Media,50.84 hours,9.3 days,4.0
1,2,Susan Morris,smorris@icloud.com,58,Female,SecureVault,19-06-2023,Technical Issue,Export feature broken since yesterday,My notes and files aren't syncing between my p...,Closed,Configuration error found and corrected. Funct...,High,Social Media,6.36 hours,2.7 days,5.0
2,3,Aaron Kim,akim@outlook.com,64,Male,ConnectHub,29-08-2023,Technical Issue,Software not loading after latest update,I synced my Google Calendar with your app last...,Closed,Configuration error found and corrected. Funct...,High,Chat,6.71 hours,1.1 days,4.0
3,4,Debra Morgan,morgan91@protonmail.com,50,Female,StreamLine CRM,08-09-2025,Technical Issue,Data not syncing between devices anymore,Two-step verification is set up on my account ...,Open,NaN,Critical,Email,0.7 hours,NaN,NaN
4,5,Julie Bailey,jbailey@yahoo.com,32,Female,OmniTrack,21-09-2023,Refund Request,Never received my order need refund now,My order was placed three weeks ago and it sti...,Pending,NaN,Medium,Phone,13.28 hours,NaN,NaN


In [87]:
## removing unnessary columns
df = df.drop(columns=['Ticket ID','Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender',
               'Date of Purchase','First Response Time','Time to Resolution','Resolution', 
               "Customer Satisfaction Rating"])

In [88]:
print(df.shape)
print(df.dtypes)


(4500, 7)
Product Purchased     str
Ticket Type           str
Ticket Subject        str
Ticket Description    str
Ticket Status         str
Ticket Priority       str
Ticket Channel        str
dtype: object


In [89]:
## combining subject and description
df['combine_text'] = df["Ticket Subject"] + " " + df["Ticket Description"]

df['combine_text'].head()

0    Service outage credit or refund request The de...
1    Export feature broken since yesterday My notes...
2    Software not loading after latest update I syn...
3    Data not syncing between devices anymore Two-s...
4    Never received my order need refund now My ord...
Name: combine_text, dtype: str

In [90]:
## changing all the text to lower and removing unneccessary spaces
df['combine_text'] = df['combine_text'].str.lower()

In [91]:
df['combine_text'] = df['combine_text'].str.strip()

In [92]:
df['combine_text'].head()

0    service outage credit or refund request the de...
1    export feature broken since yesterday my notes...
2    software not loading after latest update i syn...
3    data not syncing between devices anymore two-s...
4    never received my order need refund now my ord...
Name: combine_text, dtype: str

In [93]:
## Label encoding ticket type and ticket prioirty

# encoding ticket type

le_type = LabelEncoder()
df['ticket_type_encoded'] = le_type.fit_transform(df['Ticket Type'])

print(dict(zip(le_type.classes_, le_type.transform(le_type.classes_))))

{'Account Access': np.int64(0), 'Billing Inquiry': np.int64(1), 'Product Inquiry': np.int64(2), 'Refund Request': np.int64(3), 'Technical Issue': np.int64(4)}


In [94]:
## encoding ticket priority

le_priority = LabelEncoder()
df['ticket_priority_encoded'] = le_priority.fit_transform(df['Ticket Priority'])

print(dict(zip(le_priority.classes_, le_priority.transform(le_priority.classes_))))

{'Critical': np.int64(0), 'High': np.int64(1), 'Low': np.int64(2), 'Medium': np.int64(3)}


In [79]:
print(df[['Ticket Type', 'ticket_type_encoded', 'Ticket Priority', 'ticket_priority_encoded']].head(10))

            Ticket Type  ...  ticket_priority_encoded
0       Technical issue  ...                        0
1       Technical issue  ...                        0
2       Technical issue  ...                        2
3       Billing inquiry  ...                        2
4       Billing inquiry  ...                        2
5  Cancellation request  ...                        2
6       Product inquiry  ...                        0
7        Refund request  ...                        0
8       Technical issue  ...                        2
9        Refund request  ...                        0

[10 rows x 4 columns]


In [95]:
## saving the encoders

import joblib
joblib.dump(le_type , '../models/le_type.pkl')
joblib.dump(le_priority, "../models/le_priority.pkl")

print("encoders saved")

encoders saved


In [96]:
print("Class distribution:")
print(df['ticket_type_encoded'].value_counts())

Class distribution:
ticket_type_encoded
4    1530
1     990
0     855
3     675
2     450
Name: count, dtype: int64


In [98]:
df.head()

,Product Purchased,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Ticket Priority,Ticket Channel,combine_text,ticket_type_encoded,ticket_priority_encoded
0,CoreManage,Refund Request,Service outage credit or refund request,The device stopped working completely after ex...,Resolved,Low,Social Media,service outage credit or refund request the de...,3,2
1,SecureVault,Technical Issue,Export feature broken since yesterday,My notes and files aren't syncing between my p...,Closed,High,Social Media,export feature broken since yesterday my notes...,4,1
2,ConnectHub,Technical Issue,Software not loading after latest update,I synced my Google Calendar with your app last...,Closed,High,Chat,software not loading after latest update i syn...,4,1
3,StreamLine CRM,Technical Issue,Data not syncing between devices anymore,Two-step verification is set up on my account ...,Open,Critical,Email,data not syncing between devices anymore two-s...,4,0
4,OmniTrack,Refund Request,Never received my order need refund now,My order was placed three weeks ago and it sti...,Pending,Medium,Phone,never received my order need refund now my ord...,3,3


In [99]:
## dropping the columns we do not require any more

df = df.drop(columns=['Ticket Description'])
df.head(3)

,Product Purchased,Ticket Type,Ticket Subject,Ticket Status,Ticket Priority,Ticket Channel,combine_text,ticket_type_encoded,ticket_priority_encoded
0,CoreManage,Refund Request,Service outage credit or refund request,Resolved,Low,Social Media,service outage credit or refund request the de...,3,2
1,SecureVault,Technical Issue,Export feature broken since yesterday,Closed,High,Social Media,export feature broken since yesterday my notes...,4,1
2,ConnectHub,Technical Issue,Software not loading after latest update,Closed,High,Chat,software not loading after latest update i syn...,4,1


In [100]:
## we kept the rest of the columns for following reasons:

# combine_text — model input
# ticket_type_encoded — model target 1
# ticket_priority_encoded — model target 2
# Ticket Type — for display in Streamlit
# Ticket Priority — for display in Streamlit
# Product Purchased — useful context to display
# Ticket Subject — useful for display in results

In [101]:
## saving the new dataset

df.to_csv('../Dataset/cleaned_tickets.csv',index=False)
print("dataset saved. Shape:", df.shape)
print("columns: ", df.columns.tolist())

dataset saved. Shape: (4500, 9)
columns:  ['Product Purchased', 'Ticket Type', 'Ticket Subject', 'Ticket Status', 'Ticket Priority', 'Ticket Channel', 'combine_text', 'ticket_type_encoded', 'ticket_priority_encoded']
